# 03 — Préparation et nettoyage des données

**Projet :** Prédiction de l'abandon et de la réussite académique des étudiants  
**Étape :** Data Cleaning & Data Preparation  
**Fichier source :** `data.csv`  
**Fichier de sortie :** `data_cleaned.csv`

---

## 1. Objectif

Ce notebook prépare le jeu de données avant les analyses avancées et la modélisation.

Les opérations réalisées sont :

1. charger le fichier source ;
2. examiner sa structure initiale ;
3. supprimer les lignes entièrement vides ;
4. détecter et supprimer les doublons exacts ;
5. harmoniser les noms de colonnes et les valeurs textuelles ;
6. traiter les valeurs manquantes ;
7. contrôler le résultat et exporter le fichier préparé.

> **Principe de prudence :** les modifications sont appliquées à une copie de travail. Le fichier source `data.csv` n'est jamais écrasé.

## 2. Importation des bibliothèques

Les bibliothèques utilisées couvrent la manipulation des données et l'affichage des tableaux de contrôle.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Paramètres d'affichage
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## 3. Chargement des données

Le fichier CSV est lu avec le séparateur point-virgule (`;`). Les noms de colonnes sont ensuite normalisés en retirant les espaces et caractères parasites en début et fin de nom.

In [2]:
# Chemins des fichiers
DATA_DIR = Path("/content")
INPUT_PATH = DATA_DIR / "data.csv"
OUTPUT_PATH = DATA_DIR / "data_cleaned.csv"

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Le fichier source est introuvable : {INPUT_PATH}. "
        "Ajoutez data.csv à votre environnement Google Colab."
    )

# Chargement du fichier source
df_source = pd.read_csv(INPUT_PATH, sep=";")

# Copie de travail : le fichier source reste inchangé
df = df_source.copy()

# Normalisation des noms de colonnes
df.columns = (
    df.columns.astype(str)
    .str.replace("\\t", "", regex=True)
    .str.replace('"', "", regex=False)
    .str.strip()
)

print("Chargement terminé.")
print(f"Observations initiales : {df.shape[0]:,}")
print(f"Variables initiales    : {df.shape[1]:,}")

display(df.head())

Chargement terminé.
Observations initiales : 4,424
Variables initiales    : 37


,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,Mother's occupation,Father's occupation,Admission grade,Displaced,Educational special needs,Debtor,Tuition fees up to date,Gender,Scholarship holder,Age at enrollment,International,Curricular units 1st sem (credited),Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Curricular units 1st sem (grade),Curricular units 1st sem (without evaluations),Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.00,1,19,12,5,9,127.30,1,0,0,1,1,0,20,0,0,0,0,0,0.00,0,0,0,0,0,0.00,0,10.80,1.40,1.74,Dropout
1,1,15,1,9254,1,1,160.00,1,1,3,3,3,142.50,1,0,0,0,1,0,19,0,0,6,6,6,14.00,0,0,6,6,6,13.67,0,13.90,-0.30,0.79,Graduate
2,1,1,5,9070,1,1,122.00,1,37,37,9,9,124.80,1,0,0,0,1,0,19,0,0,6,0,0,0.00,0,0,6,0,0,0.00,0,10.80,1.40,1.74,Dropout
3,1,17,2,9773,1,1,122.00,1,38,37,5,3,119.60,1,0,0,1,0,0,20,0,0,6,8,6,13.43,0,0,6,10,5,12.40,0,9.40,-0.80,-3.12,Graduate
4,2,39,1,8014,0,1,100.00,1,37,38,9,9,141.50,0,0,0,1,0,0,45,0,0,6,9,5,12.33,0,0,6,6,6,13.00,0,13.90,-0.30,0.79,Graduate


## 4. Diagnostic initial

Avant toute transformation, un premier état des lieux permet de connaître les types de données, la complétude et le nombre de valeurs distinctes de chaque variable.

In [3]:
initial_report = pd.DataFrame({
    "Type": df.dtypes.astype(str),
    "Valeurs non nulles": df.notna().sum(),
    "Valeurs manquantes": df.isna().sum(),
    "Taux de complétude (%)": (df.notna().mean() * 100).round(2),
    "Valeurs distinctes": df.nunique(dropna=True)
})

display(initial_report)

,Type,Valeurs non nulles,Valeurs manquantes,Taux de complétude (%),Valeurs distinctes
Marital status,int64,4424,0,100.00,6
Application mode,int64,4424,0,100.00,18
Application order,int64,4424,0,100.00,8
Course,int64,4424,0,100.00,17
Daytime/evening attendance,int64,4424,0,100.00,2
Previous qualification,int64,4424,0,100.00,17
Previous qualification (grade),float64,4424,0,100.00,101
Nacionality,int64,4424,0,100.00,21
Mother's qualification,int64,4424,0,100.00,29
Father's qualification,int64,4424,0,100.00,34


## 5. Suppression des lignes entièrement vides

Les lignes ne contenant aucune information sont retirées. Cette opération ne concerne pas les lignes qui contiennent au moins une valeur renseignée.

In [4]:
rows_before_empty = len(df)

df = df.dropna(how="all").copy()

empty_rows_removed = rows_before_empty - len(df)

print(f"Lignes entièrement vides supprimées : {empty_rows_removed:,}")
print(f"Nombre de lignes après traitement  : {len(df):,}")

Lignes entièrement vides supprimées : 0
Nombre de lignes après traitement  : 4,424


## 6. Détection et traitement des doublons exacts

Les doublons sont identifiés sur l'ensemble des colonnes. Les occurrences supplémentaires sont supprimées en conservant la première occurrence.

Cette règle concerne uniquement les lignes strictement identiques ; elle ne permet pas de conclure que deux étudiants ayant des caractéristiques semblables représentent le même individu.

In [5]:
duplicates_before = int(df.duplicated().sum())

df = df.drop_duplicates(keep="first").copy()

print(f"Doublons exacts détectés et supprimés : {duplicates_before:,}")
print(f"Nombre de lignes restantes           : {len(df):,}")

Doublons exacts détectés et supprimés : 0
Nombre de lignes restantes           : 4,424


## 7. Harmonisation des valeurs textuelles

Les espaces superflus au début et à la fin des valeurs textuelles sont retirés. Les valeurs manquantes restent représentées par `NaN` afin de pouvoir être traitées correctement dans la section suivante.

In [6]:
text_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()

for column in text_columns:
    df[column] = df[column].map(
        lambda value: value.strip() if isinstance(value, str) else value
    )

print(f"Variables textuelles harmonisées : {len(text_columns)}")

Variables textuelles harmonisées : 1


## 8. Traitement des valeurs manquantes

La stratégie appliquée dépend du type de variable :

- **Variables numériques :** remplacement par la médiane ;
- **Variables catégorielles ou textuelles :** remplacement par le mode ;
- **Variable sans aucune valeur exploitable :** conservation des valeurs manquantes et signalement dans le rapport.

La médiane est moins sensible aux valeurs extrêmes que la moyenne. Le mode permet de remplacer une valeur manquante par la modalité la plus fréquente.

> **Point méthodologique :** pour une évaluation prédictive rigoureuse, les paramètres d'imputation doivent être appris uniquement sur les données d'entraînement, puis appliqués aux données de validation et de test. Le traitement global présenté ici convient à une préparation exploratoire, mais doit être intégré à un pipeline sans fuite de données avant l'entraînement final.

In [7]:
missing_before = df.isna().sum()
missing_before = missing_before[missing_before > 0]

imputation_log = []

for column in df.columns:
    missing_count = int(df[column].isna().sum())

    if missing_count == 0:
        continue

    if pd.api.types.is_numeric_dtype(df[column]):
        fill_value = df[column].median()

        if pd.isna(fill_value):
            strategy = "Aucune imputation possible : variable entièrement vide"
        else:
            df[column] = df[column].fillna(fill_value)
            strategy = f"Médiane ({fill_value:.4g})"

    else:
        modes = df[column].mode(dropna=True)

        if modes.empty:
            strategy = "Aucune imputation possible : aucune modalité disponible"
        else:
            fill_value = modes.iloc[0]
            df[column] = df[column].fillna(fill_value)
            strategy = f"Mode ({fill_value})"

    imputation_log.append({
        "Variable": column,
        "Valeurs manquantes initiales": missing_count,
        "Méthode appliquée": strategy
    })

imputation_report = pd.DataFrame(imputation_log)

if imputation_report.empty:
    print("Aucune valeur manquante à traiter.")
else:
    display(imputation_report)

Aucune valeur manquante à traiter.


## 9. Contrôle après nettoyage

Cette section vérifie l'état du dataset après les transformations et permet de comparer les dimensions initiales et finales.

In [8]:
final_report = pd.DataFrame({
    "Type": df.dtypes.astype(str),
    "Valeurs non nulles": df.notna().sum(),
    "Valeurs manquantes": df.isna().sum(),
    "Taux de complétude (%)": (df.notna().mean() * 100).round(2),
    "Valeurs distinctes": df.nunique(dropna=True)
})

display(final_report)

print("Contrôles de qualité")
print("-" * 40)
print(f"Lignes initiales                  : {len(df_source):,}")
print(f"Lignes finales                    : {len(df):,}")
print(f"Colonnes finales                  : {df.shape[1]:,}")
print(f"Doublons exacts restants           : {df.duplicated().sum():,}")
print(f"Valeurs manquantes restantes       : {df.isna().sum().sum():,}")

,Type,Valeurs non nulles,Valeurs manquantes,Taux de complétude (%),Valeurs distinctes
Marital status,int64,4424,0,100.00,6
Application mode,int64,4424,0,100.00,18
Application order,int64,4424,0,100.00,8
Course,int64,4424,0,100.00,17
Daytime/evening attendance,int64,4424,0,100.00,2
Previous qualification,int64,4424,0,100.00,17
Previous qualification (grade),float64,4424,0,100.00,101
Nacionality,int64,4424,0,100.00,21
Mother's qualification,int64,4424,0,100.00,29
Father's qualification,int64,4424,0,100.00,34


Contrôles de qualité
----------------------------------------
Lignes initiales                  : 4,424
Lignes finales                    : 4,424
Colonnes finales                  : 37
Doublons exacts restants           : 0
Valeurs manquantes restantes       : 0


## 10. Exportation du jeu de données préparé

Le dataset nettoyé est exporté dans un nouveau fichier CSV. L'index pandas n'est pas ajouté au fichier.

In [9]:
df.to_csv(OUTPUT_PATH, index=False, sep=";")

print("Exportation terminée.")
print(f"Fichier créé : {OUTPUT_PATH}")
print(f"Taille du fichier : {OUTPUT_PATH.stat().st_size / 1024:.2f} Ko")

Exportation terminée.
Fichier créé : /content/data_cleaned.csv
Taille du fichier : 516.41 Ko


## 11. Bilan de la préparation

À l'issue de ce notebook, le fichier `data_cleaned.csv` contient les données après les opérations de nettoyage décrites ci-dessus.

### Opérations réalisées

- suppression des lignes entièrement vides ;
- suppression des doublons strictement identiques ;
- harmonisation des espaces dans les valeurs textuelles ;
- imputation des valeurs manquantes numériques par la médiane ;
- imputation des valeurs manquantes catégorielles par le mode ;
- vérification de la qualité des données après traitement ;
- exportation dans un nouveau fichier CSV.

### Livrable

**`data_cleaned.csv`** — version préparée du jeu de données, destinée à la suite du processus analytique.

### Étape suivante

Poursuivre avec l'ingénierie des variables (*Feature Engineering*), l'encodage des variables catégorielles et la préparation des jeux d'entraînement, de validation et de test.